In [ ]:
# ==============================================================================
# VISISHIELD-EDGE / NFL COMBINE FSM: ALL-IN-ONE NOTEBOOK CELL SCRIPT
# ==============================================================================

from pathlib import Path
import os
import subprocess
import sys
import unittest
from datetime import datetime, timedelta, timezone

print("=== 1. STARTING REPOSITORY SETUP & PIPELINE ===")

REPOSITORY_URL = 'https://github.com/hajousfn-web/nfl-combai-edge-2027.git'
REPOSITORY_NAME = 'nfl-combai-edge-2027'
working_directory = Path.cwd().resolve()
search_roots = (working_directory, *working_directory.parents)
project_root = next(
    (candidate for candidate in search_roots if (candidate / 'src' / 'pipeline.py').is_file()),
    None,
)

if project_root is None:
    clone_directory = working_directory / REPOSITORY_NAME
    if clone_directory.exists():
        if (clone_directory / 'src' / 'pipeline.py').is_file():
            project_root = clone_directory
        else:
            raise RuntimeError(
                f'Cannot use existing clone destination without src/pipeline.py: {clone_directory}'
            )
    else:
        try:
            subprocess.run(
                ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(clone_directory)],
                check=True,
                capture_output=True,
                text=True,
            )
        except FileNotFoundError as error:
            raise RuntimeError('Git is required to clone the project repository.') from error
        except subprocess.CalledProcessError as error:
            details = error.stderr.strip() or error.stdout.strip() or str(error)
            raise RuntimeError(f'Could not clone the project repository: {details}') from error
        project_root = clone_directory

if not (project_root / 'src' / 'pipeline.py').is_file():
    raise FileNotFoundError(f'Project source tree is missing: {project_root / "src"}')

os.chdir(project_root)
print(f'Project repository ready at: {Path.cwd()}')

# VisiShield-Edge: NFL Big Data Bowl 2027 (Biomechanical Risk & Kinematics FSM)
**Offline 10 Hz tracking + nested/forward Ridge validation + statistical audit + player biomechanical telemetry.**

In [ ]:
print("=== 2. ENVIRONMENT & LOCAL PATH CONFIGURATION ===")

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'src' / 'pipeline.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the repository root containing src/pipeline.py.'
    )

SRC = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
TRACKING_CSV = DATA_DIR / 'combine_tracking.csv'
PERFORMANCE_CSV = DATA_DIR / 'player_season_outcomes.csv'
sys.path.insert(0, str(SRC))

import edge_safety_fsm
import feature_engineering
import model_integration
import pipeline
import execution_ring
import safety_kernel
import safety_engine
import statistical_audit
import visualization
import zone_loader

assert callable(feature_engineering.run_feature_pipeline)
assert callable(model_integration.run_model_pipeline)
assert callable(pipeline.run_full_pipeline)
assert callable(statistical_audit.run_statistical_audit)
assert callable(visualization.generate_project_visualizations)
assert callable(visualization.generate_audit_visualizations)
assert callable(edge_safety_fsm.EdgeSafetyFSM)
assert callable(safety_kernel.SafetyKernel)
assert callable(safety_engine.SafetyEngine)
assert callable(execution_ring.DeterministicExecutionRing)
assert callable(zone_loader.load_zone_thresholds)

print(f'Project root: {PROJECT_ROOT}')
print('All analysis, telemetry, FSM, and visualization modules imported successfully.')

## 3. Module Self-Validation (Linux Path Fixed)

In [ ]:
print("=== 3. RUNNING MODULE SELF-TESTS ===")

self_test_modules = [
    'feature_engineering.py',
    'model_integration.py',
    'pipeline.py',
    'statistical_audit.py',
    'edge_safety_fsm.py',
    'zone_loader.py',
    'execution_ring.py',
    'safety_kernel.py',
    'safety_engine.py',
    'visualization.py',
]

for module_name in self_test_modules:
    test = subprocess.run(
        [sys.executable, str(SRC / module_name), '--self-test'],
        cwd=PROJECT_ROOT,
        check=False,
        text=True,
        capture_output=True,
    )
    print(f'--- {module_name} ---')
    print(test.stdout)
    if test.stderr:
        print(test.stderr)
    if test.returncode != 0:
        raise RuntimeError(f'{module_name} self-test failed.')

print(f'All {len(self_test_modules)} module self-tests passed cleanly without errors!')

## 4. Biomechanical Risk & Telemetry Demonstration

In [ ]:
print("=== 4. RUNNING TELEMETRY & BIOMECHANICAL RISK FSM DEMO ===")

from zone_loader import ZoneThresholds

mock_thresholds = ZoneThresholds(
    warning_enter_m=1.1,
    warning_clear_m=1.2,
    danger_enter_m=0.8,
    danger_clear_m=0.9,
    emergency_enter_m=0.4,
    emergency_clear_m=0.5,
)
engine = safety_engine.SafetyEngine(
    thresholds=mock_thresholds,
    ring_capacity=8,
    recovery_samples_required=1,
    reset_safe_samples_required=2,
    zone_id='illustrative-player-risk',
)
base_timestamp_ns = 1_798_000_000_000_000_000
demo_readings = [0.3, 1.3, 1.3]
for index, distance_m in enumerate(demo_readings):
    engine.submit_reading(
        timestamp_ns=base_timestamp_ns + index * 100_000_000,
        distance_m=distance_m,
        speed_mps=2.0,
        acceleration_mps2=0.1,
        cut_efficiency=0.9,
    )
    event = engine.process_next()
    print("FSM State Event:", event.as_dict())

reset = engine.manual_reset(
    timestamp_ns=base_timestamp_ns + 300_000_000,
    operator_id='nfl-reviewer',
    safe_to_reset=True,
)
print('Manual telemetry reset acknowledgement:', reset.as_dict())
print('Execution latency metrics (microseconds):', safety_engine.benchmark_processing(iterations=100, thresholds=mock_thresholds))
print("=== PIPELINE & NOTEBOOK EXECUTION COMPLETED SUCCESSFULLY ===")